# Native CLM v0 — JAM Learning Demo

This notebook starts from the canonical Native CLM v0 M1 checkpoint, post-trains it on `jam-knowledge-v0.1`, evaluates before/after JAM and TinyStories metrics, publishes the JAM model under `archelabs-org/native-clm-v0/jam-v0.1/`, and pushes lightweight logs back to GitHub.

This is an engineering/demo run, not a replay-free continual-learning decision.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

BRANCH = "codex/native-clm-jam-demo-v0.1"
REPO = Path("/kaggle/working/mini-cells")
BASE_DATA = Path("/kaggle/working/native-clm-jam-base-data")
OUT = REPO / "artifacts/demos/native-clm-jam-v0.1"

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--branch", BRANCH, "https://github.com/ArcheLabs/mini-cells.git", REPO])
os.chdir(REPO)
run(["git", "fetch", "origin"])
run(["git", "checkout", BRANCH])
run(["git", "pull", "--ff-only", "origin", BRANCH])
run([sys.executable, "-m", "pip", "install", "-e", ".[dev,lm]"])
run([sys.executable, "-m", "pip", "install", "huggingface_hub>=0.28"])

import torch
print("branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["GITHUB_TOKEN"] = secrets.get_secret("GITHUB_TOKEN")
assert os.environ["HF_TOKEN"], "Missing Kaggle Secret: HF_TOKEN"
assert os.environ["GITHUB_TOKEN"], "Missing Kaggle Secret: GITHUB_TOKEN"
print("Secrets loaded (values not printed).")


In [ ]:
# Prepare a small deterministic TinyStories rehearsal/regression cache.
run([
    sys.executable,
    "scripts/research/prepare_native_clm_v0_m1_data.py",
    "--dataset-id", "roneneldan/TinyStories",
    "--train-docs", "5000",
    "--validation-docs", "1000",
    "--output-dir", BASE_DATA,
])
print((BASE_DATA / "manifest.json").read_text())


In [ ]:
# Run the complete before -> JAM post-training -> after pipeline and publish weights to HF.
run([
    sys.executable,
    "scripts/research/run_native_clm_jam_demo.py",
    "--base-train-file", BASE_DATA / "train.txt",
    "--base-validation-file", BASE_DATA / "validation.txt",
    "--device", "cuda",
    "--steps", "1200",
    "--precision", "fp16",
    "--publish-hf",
])


In [ ]:
bench = json.loads((OUT / "benchmarks.json").read_text())
prov = json.loads((OUT / "provenance.json").read_text())
print(json.dumps({
    "base_sha256": prov["base_checkpoint_sha256"],
    "jam_sha256": prov["final_checkpoint_sha256"],
    "selected_step": prov["selected_step"],
    "validation_before": bench["before"]["validation"],
    "validation_after": bench["after"]["validation"],
    "reasoning_before": bench["before"]["reasoning"],
    "reasoning_after": bench["after"]["reasoning"],
    "base_before": bench["before"]["base"],
    "base_after": bench["after"]["base"],
}, indent=2))
print("\n--- Q&A LOG ---\n")
print((OUT / "QA_LOG.md").read_text())


In [ ]:
# Push only lightweight benchmark/provenance/Q&A artifacts to the research branch.
run([
    sys.executable,
    "scripts/research/publish_native_clm_jam_demo.py",
    "--branch", BRANCH,
])
print("Published GitHub demo artifacts and Hugging Face jam-v0.1 weights.")
